# Step 3 — Train Gaussian Splatting

The COLMAP work is already done: steps 1 and 2 produced a folder holding
`images/` and `sparse/0/`. This notebook only trains.

**Before you run anything:** `Runtime` → `Change runtime type` → **T4 GPU**.

Rough timings on a free-tier T4: setup 10 minutes, training 45-70 minutes.

Every checkpoint is copied to Drive the moment it appears, so if Colab cuts the
session off you keep whatever finished.

## 1. GPU

In [ ]:
import subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU attached. Runtime > Change runtime type > T4 GPU.")

print("GPU:", gpu.stdout.strip())

# CUDA compute capability, needed when compiling the rasteriser.
name = gpu.stdout.lower()
if "t4" in name:
    ARCH = "7.5"
elif "l4" in name:
    ARCH = "8.9"
elif "a100" in name:
    ARCH = "8.0"
else:
    ARCH = "7.5"
    print("Unknown card — assuming 7.5. If compilation fails, set ARCH by hand.")
print("ARCH =", ARCH)

## 2. Google Drive

In [ ]:
import os

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
print("Drive mounted.")

## 3. Configuration — edit this cell

In [ ]:
# ==================== EDIT THIS CELL ====================

# The COLMAP result from step 2. Either works:
#   - a folder on Drive holding images/ and sparse/0/
#   - a .tar.gz of that same folder
COLMAP_INPUT = "/content/drive/MyDrive/img3dpl/meo_no"

# Short name, used for output filenames.
PROJECT = "meo_no"

# Folder on Drive for the finished .ply files.
OUTPUT_DIR = "/content/drive/MyDrive/img3dpl/results"

# Checkpoints to save. Drop the later ones if Colab keeps cutting you off.
SAVES = [7000, 15000, 30000]

# True adds two flags that slow down how fast new Gaussians are created.
# Needed on detailed scenes (over 100,000 starting points) because a T4 only
# has 15 GB of VRAM.
LIMIT_VRAM = True

# Spherical-harmonics degree for the compressed copy: 1 is light,
# 3 keeps every reflection but stays large.
COMPRESS_SH_DEGREE = 1

# ========================================================
import os

ITERATIONS = max(SAVES)
WORK = f"/content/{PROJECT}"

if not os.path.exists(COLMAP_INPUT):
    raise SystemExit(f"Not found on Drive: {COLMAP_INPUT}")

print("Input      :", COLMAP_INPUT)
print("Output     :", OUTPUT_DIR)
print("Checkpoints:", SAVES)

## 4. Load the COLMAP result

Handles both shapes of input: a plain folder, or a `.tar.gz`. If the archive has
a single wrapper folder inside, it descends into it automatically.

In [ ]:
import os
import shutil
import tarfile
import time

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)
started = time.time()

if os.path.isdir(COLMAP_INPUT):
    print("Input is a folder — copying to local disk...")
    # Copy rather than work straight off Drive: training reads these files
    # thousands of times, and Drive is far slower than local disk.
    shutil.copytree(COLMAP_INPUT, WORK, dirs_exist_ok=True)
else:
    print("Input is an archive — extracting...")
    with tarfile.open(COLMAP_INPUT) as archive:
        try:
            archive.extractall(WORK, filter="data")
        except TypeError:
            # Python older than 3.12 has no filter argument.
            archive.extractall(WORK)

# Some archives wrap everything in one folder. Step into it if so.
if not os.path.isdir(f"{WORK}/images"):
    inner = [d for d in os.listdir(WORK) if os.path.isdir(f"{WORK}/{d}")]
    if len(inner) == 1 and os.path.isdir(f"{WORK}/{inner[0]}/images"):
        WORK = f"{WORK}/{inner[0]}"
        print("Descended into:", inner[0])

print(f"Done in {time.time() - started:.0f}s")
print("Contents:", sorted(os.listdir(WORK)))

if not os.path.isdir(f"{WORK}/images"):
    raise SystemExit("No images/ folder. Did step 2 finish?")
if not os.path.isdir(f"{WORK}/sparse/0"):
    raise SystemExit("No sparse/0 folder. Did step 2 finish?")

print("Images:", len(os.listdir(f"{WORK}/images")))
print("Sparse:", sorted(os.listdir(f"{WORK}/sparse/0")))

## 5. Install Gaussian Splatting

Takes about 10 minutes, most of it compiling the CUDA rasteriser. Output is kept
quiet unless something fails, in which case the last 30 lines are printed.

In [ ]:
import os
import subprocess

os.environ["TORCH_CUDA_ARCH_LIST"] = ARCH
REPO = "/content/gaussian-splatting"


def step(label, command):
    """Run a shell command quietly. On failure, show enough to diagnose it."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FAILED: {label}\n")
        print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))
        raise RuntimeError(label)
    print("ok  ", label)


if not os.path.isdir(REPO):
    step("clone repository",
         "git clone -q --recursive "
         f"https://github.com/graphdeco-inria/gaussian-splatting {REPO}")

step("plyfile", "pip -q install plyfile")
step("diff-gaussian-rasterization",
     f"pip -q install {REPO}/submodules/diff-gaussian-rasterization")
step("simple-knn", f"pip -q install {REPO}/submodules/simple-knn")

if os.path.isdir(f"{REPO}/submodules/fused-ssim"):
    step("fused-ssim", f"pip -q install {REPO}/submodules/fused-ssim")

import torch
from diff_gaussian_rasterization import GaussianRasterizer  # noqa: F401

print()
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Ready to train.")

## 6. Train

Each checkpoint is copied to Drive as soon as it lands on disk. If Colab drops
the session halfway, everything already copied is still yours.

A progress line prints every 30 seconds so you can tell it is alive.

In [ ]:
import glob
import os
import shutil
import subprocess
import time

os.makedirs(OUTPUT_DIR, exist_ok=True)

command = ["python", "train.py",
           "-s", WORK,
           "-m", f"{WORK}/output",
           "--iterations", str(ITERATIONS),
           "--save_iterations"] + [str(s) for s in SAVES]

if LIMIT_VRAM:
    command += ["--densify_grad_threshold", "0.0004",
                "--densify_until_iter", "12000"]

log_path = f"{WORK}/train.log"
log_file = open(log_path, "w")
trainer = subprocess.Popen(command, cwd="/content/gaussian-splatting",
                           stdout=log_file, stderr=subprocess.STDOUT, text=True)
print("Training started. A progress line follows every 30 seconds.\n")

copied = set()


def copy_new_checkpoints():
    """Copy any checkpoint that has appeared since the last look."""
    pattern = f"{WORK}/output/point_cloud/iteration_*/point_cloud.ply"
    for path in glob.glob(pattern):
        step_number = path.split("iteration_")[1].split("/")[0]
        if step_number in copied:
            continue
        time.sleep(5)          # let the file finish being written
        target = f"{OUTPUT_DIR}/{PROJECT}_{step_number}.ply"
        shutil.copy(path, target)
        copied.add(step_number)
        size = os.path.getsize(target) / 1e6
        print(f">>> SAVED TO DRIVE: {PROJECT}_{step_number}.ply ({size:.0f} MB)")


while trainer.poll() is None:
    time.sleep(30)
    copy_new_checkpoints()
    tail = subprocess.run(["tail", "-1", log_path],
                          capture_output=True, text=True).stdout.strip()
    if tail:
        print(tail)

log_file.close()
copy_new_checkpoints()

if not copied:
    print("\nNo checkpoint was produced. Last 30 lines of the log:\n")
    print(subprocess.run(["tail", "-30", log_path],
                         capture_output=True, text=True).stdout)
else:
    print("\nFinished. Saved:", sorted(copied, key=int))

## 7. Make a lighter copy

Throws away the near-transparent blobs and lowers the spherical-harmonics degree
from 3 to `COMPRESS_SH_DEGREE`. The lighter file sits beside the original on
Drive, ending in `_light.ply`.

Useful because the full file often will not open on a modest laptop.

In [ ]:
import glob
import os

import numpy as np
from plyfile import PlyData, PlyElement


def compress_ply(source, target, sh_degree=1, opacity_threshold=0.05):
    """Drop faint Gaussians and trim the colour detail. Returns (kept, total)."""
    vertices = PlyData.read(source)["vertex"]

    # Stored opacity is pre-sigmoid, so convert before comparing.
    opacity = 1 / (1 + np.exp(-np.asarray(vertices["opacity"])))
    keep = opacity > opacity_threshold

    rest_per_channel = {0: 0, 1: 3, 2: 8, 3: 15}[sh_degree]

    src_names = ["x", "y", "z", "nx", "ny", "nz", "f_dc_0", "f_dc_1", "f_dc_2"]
    dst_names = list(src_names)

    index = 0
    for channel in range(3):
        for i in range(rest_per_channel):
            src_names.append(f"f_rest_{channel * 15 + i}")
            dst_names.append(f"f_rest_{index}")
            index += 1

    tail = ["opacity", "scale_0", "scale_1", "scale_2",
            "rot_0", "rot_1", "rot_2", "rot_3"]
    src_names += tail
    dst_names += tail

    out = np.empty(int(keep.sum()), dtype=[(n, "f4") for n in dst_names])
    for src, dst in zip(src_names, dst_names):
        out[dst] = np.asarray(vertices[src])[keep]

    PlyData([PlyElement.describe(out, "vertex")]).write(target)
    return int(keep.sum()), len(keep)


for source in sorted(glob.glob(f"{OUTPUT_DIR}/{PROJECT}_*.ply")):
    if "_light" in source:
        continue
    target = source.replace(".ply", "_light.ply")
    kept, total = compress_ply(target=target, source=source,
                               sh_degree=COMPRESS_SH_DEGREE)
    before = os.path.getsize(source) / 1e6
    after = os.path.getsize(target) / 1e6
    print(f"{os.path.basename(target)}: kept {kept}/{total} blobs | "
          f"{before:.0f} -> {after:.0f} MB ({before / after:.1f}x smaller)")

## 8. View the result

Download a `_light.ply` from Drive, open **https://superspl.at/editor**, and drag
the file in.

The model has no real-world scale — use the editor's scale tool to size it by eye.

---

### If something went wrong

**Out of VRAM during training** — set `LIMIT_VRAM = True` if it is not already,
or drop the last entry from `SAVES`.

**The light copy looks flat, reflections gone** — set `COMPRESS_SH_DEGREE = 3`
and run cell 7 again. The full file is still on Drive; no need to retrain.

**Colab disconnected halfway** — run again from cell 1. Checkpoints already
copied to Drive are still good.

**Only a few images were reconstructed in step 2** — the problem is the photos,
not the training. Neighbouring shots need roughly 60-80% overlap.